# Interactive simulation checks: LBWSG (birth weight & gestational age)

Verifies that IFA and MMS shift the birth-weight and gestational-age *birth-exposure*
pipelines by the artifact's excess-shift amounts, and that IFA reduces preterm birth.
Ported from the research portfolio VnV notebook `model_23.0_interactive_simulation_lbwsg_post_bugfix`;
updated to the current Engine (`vivarium.engine`) API and generalized from the source's
single-simulant checks to all simulants.

Dropped from the source (add back as needed): the exploratory PTB-prevalence reconstruction
(depended on an external /snfs1 st-gpr file and documented a known ANC/LBWSG-correlation
miscalibration); the ACS-window and GA-error-by-ultrasound checks (covered in the acs and
facility_choice notebooks); the MMS gestational-age subpop checks (MMS GA effects are covered
in effect_propagation); and the LBWSG relative-risk interpolator check (its
`..._on_all_causes.relative_risk` pipeline no longer exists -- the model now uses per-cause
LBWSG RRs, so re-add against one of those).

In [ ]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

import numpy as np
import pandas as pd
from pathlib import Path

import vivarium_gates_mncnh
from vivarium.artifact import Artifact
from vivarium.engine import InteractiveContext
from vivarium.engine.framework.configuration import build_model_specification

In [ ]:
!pip list | grep vivarium

In [ ]:
SPEC_PATH = Path(vivarium_gates_mncnh.__file__).parent / "model_specifications/model_spec.yaml"
BIRTH_PIPELINES = ["birth_weight.birth_exposure", "gestational_age.birth_exposure"]
KEEP = ["oral_iron_intervention", "anc_attendance", "pregnancy_outcome", "sex_of_child"]
# The per-axis birth exposures come from the combined LBWSG birth-exposure pipeline
# (a DataFrame with 'birth_weight'/'gestational_age' columns). The IFA/MMS effects modify
# this pipeline, so comparing it before vs after ANC captures the applied shift. We expose
# the two axes under the BIRTH_PIPELINES names so the checks below read naturally.
AXIS = {"birth_weight.birth_exposure": "birth_weight", "gestational_age.birth_exposure": "gestational_age"}

def frame(sim):
    pop = sim.get_population(KEEP)
    be = sim.get_population("low_birth_weight_and_short_gestation.birth_exposure")
    for name, axis in AXIS.items():
        pop[name] = be[axis]
    return pop

def run_and_capture(scenario=None):
    """Return (initial-exposure frame, post-ANC frame) for a scenario."""
    spec = build_model_specification(SPEC_PATH)
    del spec.configuration.observers
    spec.configuration.population.population_size = 20_000 * 10
    if scenario is not None:
        spec.configuration.intervention.scenario = scenario
    sim = InteractiveContext(spec)
    initial = frame(sim)  # before any ANC / IFA effect
    get_event_name = sim._builder.time.simulation_event_name()
    while get_event_name() != "delivery_facility":  # past all ANC + ultrasound
        sim.step()
    return initial, frame(sim)

In [ ]:
base_spec = build_model_specification(SPEC_PATH)
art = Artifact(base_spec.configuration.input_data.artifact_path)
draw = "draw_" + str(base_spec.configuration.input_data.input_draw_number)

# Artifact excess shifts (index/order mirrors the source VnV notebook).
ifa_excess = art.load("risk_factor.iron_folic_acid_supplementation.excess_shift")[draw]
ifa_bw_shift = ifa_excess[1]
ifa_ga_shift = ifa_excess.tail(1).values[0]
mms_bw_shift = art.load("risk_factor.multiple_micronutrient_supplementation.excess_shift")[draw][1]
ifa_bw_shift, ifa_ga_shift, mms_bw_shift

In [ ]:
base_init, base_final = run_and_capture()
base_final[["oral_iron_intervention", "anc_attendance"] + BIRTH_PIPELINES].head()

## IFA shifts birth weight and gestational age (baseline)

In [ ]:
# For simulants covered by IFA, the birth-weight and gestational-age birth exposures rise by
# exactly the artifact excess shifts; for the untreated, they don't move.
bw_shift = base_final["birth_weight.birth_exposure"] - base_init["birth_weight.birth_exposure"]
ga_shift = base_final["gestational_age.birth_exposure"] - base_init["gestational_age.birth_exposure"]
ifa = base_final.oral_iron_intervention == "ifa"

assert np.allclose(bw_shift[ifa], ifa_bw_shift, atol=1e-6), "IFA birth-weight shift != artifact excess shift"
assert np.allclose(bw_shift[~ifa], 0, atol=1e-6), "untreated simulants had a birth-weight shift"
assert np.allclose(ga_shift[ifa], ifa_ga_shift, atol=1e-6), "IFA gestational-age shift != artifact excess shift"
assert np.allclose(ga_shift[~ifa], 0, atol=1e-6), "untreated simulants had a gestational-age shift"

## MMS shifts birth weight relative to baseline (`mms_total_scaleup`)

In [ ]:
# Common random numbers -> same simulants. Under MMS, birth weight rises relative to baseline by
# the MMS excess shift (for those already on IFA at baseline) or MMS + IFA (for the baseline-untreated).
_, mms_final = run_and_capture("mms_total_scaleup")
bw_delta = mms_final["birth_weight.birth_exposure"] - base_final["birth_weight.birth_exposure"]
mms_cov = mms_final.oral_iron_intervention == "mms"
baseline_ifa = base_final.oral_iron_intervention == "ifa"

assert np.allclose(bw_delta[mms_cov & baseline_ifa], mms_bw_shift, atol=1e-6), \
    "MMS-over-IFA birth-weight shift != artifact MMS excess shift"
assert np.allclose(bw_delta[mms_cov & ~baseline_ifa], mms_bw_shift + ifa_bw_shift, atol=1e-6), \
    "MMS-over-untreated birth-weight shift != MMS + IFA excess shift"

## IFA reduces preterm birth

In [ ]:
# Raising gestational age should lower the pipeline preterm-birth rate from initialization to post-ANC.
init_preterm = (base_init["gestational_age.birth_exposure"] < 37).mean()
final_preterm = (base_final["gestational_age.birth_exposure"] < 37).mean()
assert final_preterm < init_preterm, \
    f"pipeline preterm rate did not fall after ANC/IFA ({init_preterm:.3f} -> {final_preterm:.3f})"